In [9]:
# llm_list = ['gemma2','llama3.1']
# for llm_model in llm_list:
#   !ollama pull {llm_model}

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling ff1d1fc78170... 100% ▕████████████████▏ 5.4 GB                         
pulling 109037bec39c... 100% ▕████████████████▏  136 B                         
pulling 097a36493f71... 100% ▕████████████████▏ 8.4 KB                         
pulling 2490e7468436... 100% ▕████████████████▏   65 B                         
pulling 10aa81da732e... 100% ▕████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 667b0c1932bc...  14% ▕██              ▏ 666 MB/4


pulling 667b0c1932bc...  14% ▕██              ▏ 671 MB/4.9 GB  4.4 MB/s   16m9spulling manifest 
pulling 667b0c1932bc...  14% ▕██              ▏ 671 MB/4.9 GB  4.4 MB/s   16m9spulling manifest 
pulling 667b0c1932bc...  14% ▕██              ▏ 672 MB/4.9 GB  4.4 MB/s   16m9spulling manifest 
pulling 667b0c1932bc...  14% ▕██              ▏ 672 MB/4.9 GB  4.4 MB/s   16m9spulling manifest 
pulling 667b0c1932bc...  14% ▕██              ▏ 673 MB/4.9 GB  4.4 MB/s   16m9spulling manifest 
pulling 667b0c1932bc...  14% ▕██              ▏ 673 MB/4.9 GB  4.4 MB/s   16m9spulling manifest 
pulling 667b0c1932bc...  14% ▕██              ▏ 674 MB/4.9 GB  4.4 MB/s   16m8spulling manifest 
pulling 667b0c1932bc...  14% ▕██              ▏ 674 MB/4.9 GB  4.4 MB/s   16m8spulling manifest 
pulling 667b0c1932bc...  14% ▕██              ▏ 675 MB/4.9 GB  4.7 MB/s  14m59spulling manifest 
pulling 667b0c1932bc...  14% ▕██              ▏ 676 MB/4.9 GB  4.7 MB/s  14m59spulling manifest 
pulling 667b0c1932bc...  14% 

In [ ]:
import time
import os
import json
import pandas as pd
from tqdm import tqdm
from llm5w1h import NewsArticle
from llm5w1h import LLMConnector
from llm5w1h import NewsAnalyzer

model = "llama3.1"
# model = "gemma2"


def extract_texts(file_path, data_list):
    # extracts specified component from json object
    def extract_component(data, component):
        annotations = data["fiveWoneH"][component]["annotated"]
        texts = [item.get("text") for item in annotations]
        return "; ".join(text for text in texts if text is not None)

    # loads json object from file
    with open(file_path, "r") as file:
        data = json.load(file)

    # extracts true components from the annotated articles (in a json object)
    text = data["text"]
    what_true = extract_component(data, "what")
    where_true = extract_component(data, "where")
    when_true = extract_component(data, "when")
    who_true = extract_component(data, "who")
    why_true = extract_component(data, "why")
    how_true = extract_component(data, "how")

    analyzer = NewsAnalyzer("http://localhost:11434","api_key", model)
    article = NewsArticle(data.get("title"), data.get("description"), text, data.get("date_publish"), data.get("url"))
    analyzer.process_article(article)

    extracted_components = analyzer.extract_components()
    # what_pred = analyzer.identify_component("What")
    # where_pred = analyzer.identify_component("Where")
    # when_pred = analyzer.identify_component("When")
    # who_pred = analyzer.identify_component("Who")
    # why_pred = analyzer.identify_component("Why")
    # how_pred = analyzer.identify_component("How")

    data_list.append({
        "text": text,
        "what_true": what_true,
        "where_true": where_true,
        "when_true": when_true,
        "who_true": who_true,
        "why_true": why_true,
        "how_true": how_true,
        "what_pred": extracted_components["what_pred"],
        "where_pred": extracted_components["where_pred"],
        "when_pred": extracted_components["when_pred"],
        "who_pred": extracted_components["who_pred"],
        "why_pred": extracted_components["why_pred"],
        "how_pred": extracted_components["how_pred"]
    })

In [15]:
data_list = []
extract_texts("./data_samples/0e5fa7c0e6252bfeeea5e3840c6cb503f299c19d24331c4ba60c5974.json", data_list)
print(data_list)

Date: 2016-11-08 14:44:34
Title: While the U.S. talks about election, UK outraged over Toblerone chocolate
Description: Some are even blaming Brexit.
Text: Skip Ad Ad Loading... x Embed x Share Toblerone is facing a mountain of criticism for changing the shape of its famous triangular candy bars in British stores, a move it blames on rising costs. USA TODAY Toblerone chocolate bars come in a variety of sizes, but recently changed the shape of two of its smaller bars sold in the UK. (Photo: Martin Ruetschi, AP) The UK has a chocolate bar crisis on its hands: the beloved Swiss chocolate bar is unrecognizable. Toblerone, the classic chocolate bar with almond-and-honey-filled triangle chunks, recently lost weight. In two sizes, the triangles shrunk, leaving wider gaps of chocolate. Toblerone can you tell me what this is all about... looks like there's half a bar missing! pic.twitter.com/C2VD3DjppE -- Alana Cartwright (@AlanaCartwrigh3) October 29, 2016  @HelenRyles Hi Helen, yes this is ju

In [16]:
pd.DataFrame(data_list)

,text,what_true,where_true,when_true,who_true,why_true,how_true,what_pred,where_pred,when_pred,who_pred,why_pred,how_pred
0,Skip Ad Ad Loading... x Embed x Share Tobleron...,outraged over Toblerone chocolate,in British stores; UK,"November 8, 2016",UK; Toblerone,changing the shape of its famous triangular ca...,changed the shape,Toblerone changed the shape of its smaller cho...,UK (specifically British stores),"October 29, 2016 (when the change was first no...","Toblerone, a Swiss chocolate company",Due to rising costs for ingredients,The company reduced the weight of two smaller ...


In [18]:
%pip install bert_score

  Using cached bert_score-0.3.13-py3-none-any.whl.metadata (15 kB)
  Using cached torch-2.6.0-cp312-cp312-win_amd64.whl.metadata (28 kB)
  Using cached transformers-4.48.2-py3-none-any.whl.metadata (44 kB)
  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached matplotlib-3.10.0-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached filelock-3.17.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.5-py3-none-any.whl.metadata (2.6 kB)
  Using cached fsspec-2025.2.0-py3-none-any.whl.metadata (11 kB)
  Using cached setuptools-75.8.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached huggingface_hub-0.28.1-py3-none-any.whl.metadata (13 kB)
  Using cached PyYAML-6.0.2-cp312-cp312-win_amd64.whl.metadata (2.1 kB)
  Using cached regex-2024.11.6-cp312-cp312-win_amd64.whl

In [19]:
from bert_score import score

c:\Users\luqui\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
def evaluate(data_list):
    e = []
    for article in data_list:
        cands = [article[component] for component in article if component.endswith("_pred")]
        refs = [article[component] for component in article if component.endswith("_true")]
        # start = time.time()
        P, R, F1 = score(cands, refs, lang="en")
        # end = time.time()
        print(F1)
        print(f"System level F1 score: {F1.mean():.3f}")
        # print("Tempo: ", end-start)
        e.append([article, cands, refs, P, R, F1])

    return e

In [23]:
evaluate(data_list)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tensor([0.8680, 0.8872, 0.9155, 0.8451, 0.8311, 0.7976])
System level F1 score: 0.857


[[{'text': 'Skip Ad Ad Loading... x Embed x Share Toblerone is facing a mountain of criticism for changing the shape of its famous triangular candy bars in British stores, a move it blames on rising costs. USA TODAY Toblerone chocolate bars come in a variety of sizes, but recently changed the shape of two of its smaller bars sold in the UK. (Photo: Martin Ruetschi, AP) The UK has a chocolate bar crisis on its hands: the beloved Swiss chocolate bar is unrecognizable. Toblerone, the classic chocolate bar with almond-and-honey-filled triangle chunks, recently lost weight. In two sizes, the triangles shrunk, leaving wider gaps of chocolate. Toblerone can you tell me what this is all about... looks like there\'s half a bar missing! pic.twitter.com/C2VD3DjppE -- Alana Cartwright (@AlanaCartwrigh3) October 29, 2016  @HelenRyles Hi Helen, yes this is just our smaller bar. -- Toblerone (@Toblerone) October 31, 2016  The 400-gram bar was reduced to a 360-gram bar and the 170-gram was reduced to 

In [24]:
# extracting components for all articles and writing in a spreadsheet
data_list = []
data_folder = './data_samples/'
cnt = 1
for filename in tqdm(os.listdir(data_folder)):
    # print(f"\n{cnt}")
    file_path = os.path.join(data_folder, filename)
    # start = time.time()
    extract_texts(file_path, data_list)
    # end = time.time()
    # print("Tempo: ", end-start)
    cnt += 1
    # if len(data_list) > 5: break
    
df = pd.DataFrame(data_list)
df.to_excel('news_5w1h_llama3.xlsx', index=False)
df.to_csv('news_5w1h_llama3.csv', index=False, encoding='utf-8')

  0%|          | 0/96 [00:00<?, ?it/s]

Date: 2016-11-06 00:00:00
Title: Comey to country: The jury will disregard...
Description: "Never mind," the FBI director says on Clinton's emails.
Text: FBI Director James B. Comey. (MICHAEL REYNOLDS/European Pressphoto Agency) If you have ever watched a procedural crime drama, you probably recognize the words, "the jury will disregard." It is the instruction judges give jurors to ignore inadmissible testimony after it has already been offered in open court. Of course the jury, composed of human beings, cannot forget what it has already heard -- even if they try. The integrity of the proceedings have already been damaged. Director James B. Comey announced Sunday that the FBI's sweep through a fresh cache of emails related to Hillary Clinton's private server found ... nothing big -- the agency concluded once again that the Democratic nominee does not deserve to be charged with a crime. The news comes a little over a week after he revealed that the FBI had found the email cache -- and s

  1%|          | 1/96 [00:31<49:07, 31.03s/it]

Date: 2016-11-09 03:35:23
Title: Lady Gaga stages protest outside Trump Tower after US election result
Description: There was misery for Hillary Clinton's band of Hollywood megafans as Donald Trump emerged victorious.
Text: After the defeat Hillary superfan Katy Perry said on Twitter 'we are not a nation that will let HATE lead us' 'Sometimes you just have to agree to disagree and grab some p***y', she tweeted before it was swiftly deleted Demi Lovato was forced to apologize for making an insensitive election joke at Trump's expense Scores of celebrities who support Hillary Clinton took to Twitter to express their dismay at the result She was seen clinging to a sanitation truck with a sign 'Love Trumps Hate' There was misery for Hillary Clinton's band of Hollywood megafans as Donald Trump emerged victorious as American President. Lady Gaga staged a protest against Trump outside Trump Tower, clinging to a sanitation truck with a sign 'Love Trumps Hate'. She was then pictured looking emo

  2%|▏         | 2/96 [01:12<57:48, 36.89s/it]

...

Date: 2016-11-03 05:10:42
Title: Chicago Cubs win World Series for first time in 108 years in dramatic 10th innings showdown
Description: The Cubs won 8-7 in an extra innings delayed by rain as thousands gathered outside Wrigley Field
Text: Get daily updates directly to your inbox + Subscribe Thank you for subscribing! Could not subscribe, try again later Invalid Email The Chicago Cubs beat the Cleveland Indians to win their first World Series in 108 years. The Cubs won 8-7 after the deciding game needed an extra innings which was delayed for rain and ended at almost midnight on Wednesday. The dramatic victory brought an end to the longest title drought in North American professional sports. The triumph of Chicago's beloved Cubbies set off a wild celebration in the streets of the Windy City after more than a century of pent up frustration for fans since their last Major League Baseball championship in 1908. MVP Zobrist, who helped the Kansas City Royals to the championship last year, s

 11%|█▏        | 11/96 [04:38<27:36, 19.48s/it]

Date: 2016-11-10 19:44:00
Title: Explosion and gunfire outside German consulate in Afghanistan
Description: AN EXPLOSION outside the German consulate in the city of Mazar-i-Sharif in northern Afghanistan has killed two people.
Text: Explosion and gunfire reported outside German consulate in Afghanistan The blast was followed by gunfire according to reports on the ground. A car laden with explosives apparently initiated the attack at the compound's security wall, followed by gunfire and an explosion inside the hotel. There have been at least two fatalities and 84 people have been taken to hospital. The attack happened at around 11:30pm local time, close to the consulate and a hotel as well as several guesthouses.
URL: http://www.express.co.uk/news/world/731052/German-consulate-explosion-gunfire-Afghanistan


 11%|█▏        | 11/96 [04:53<37:49, 26.70s/it]


KeyboardInterrupt: 

In [28]:
df = pd.DataFrame(data_list)
df.to_excel('news_5w1h_llama3.xlsx', index=False)
df.to_csv('news_5w1h_llama3.csv', index=False, encoding='utf-8')

In [29]:
df

,text,what_true,where_true,when_true,who_true,why_true,how_true,what_pred,where_pred,when_pred,who_pred,why_pred,how_pred
0,FBI Director James B. Comey. (MICHAEL REYNOLDS...,Clinton's emails,,Sunday,James B. Comey; Clinton,the new email cache may have nothing of releva...,vague note,The FBI's review of new emails related to Hill...,USA (specifically Pennsylvania and Ohio),Sunday (2016-11-06),"FBI Director James B. Comey, Hillary Clinton, ...",Not mentioned,"The FBI reviewed the material quickly, suggest..."
1,After the defeat Hillary superfan Katy Perry s...,protest outside Trump Tower after US election ...,New York City,the early hours of Wednesday,Lady Gaga; Katy Perry; Demi Lovato,Donald Trump emerged victorious,stages protest,Celebrities stage a protest and express their ...,"Trump Tower, New York City",2016-11-09,"Lady Gaga, Katy Perry, Demi Lovato, Eva Longor...",The reason behind the event is not explicitly ...,The protest was staged outside Trump Tower in ...
2,Her campaign said the seemingly positive outco...,Blames F.B.I. Director for Election Loss,Saturday,,Hillary Clinton,he had revived the inquiry into her use of a p...,Blames,Hillary Clinton blaming F.B.I. Director James ...,United States (battleground states),2016-11-12,"Hillary Clinton, James B. Comey",The announcement by James B. Comey that he had...,The F.B.I.'s examination of new emails and Mr....
3,The death toll from a powerful Taliban truck b...,attacks German consulate; truck bombing,northern Afghan city of Mazar-i-Sharif,late Thursday,Taliban,revenge attack,rammed his explosives-laden; rammed,Truck bombing at the German consulate in Mazar...,"Mazar-i-Sharif city, northern Afghanistan",Thursday (late) and Friday (at least six death...,Taliban,Revenge attack for US air strikes in Kunduz pr...,A suicide attacker rammed his explosives-laden...
4,Police investigating the Croydon tram crash ha...,name final three victims; died,Croydon,Wednesday,"Donald Collett, Philip Logan and Robert Huxley...",above the permitted speed; tram crash that kil...,high speed; crash,Croydon tram crash that killed seven people,"Sandilands tram stop in Croydon, south London","Wednesday (no specific date mentioned), but th...","Police, tram driver, passengers, and victims i...","Not mentioned explicitly, but the investigatio...",The tram overturned as it entered a bend at hi...
5,It's a triangle offense. Toblerone is facing a...,are pissed,UK,,Toblerone; People,higher costs for numerous ingredients; new can...,criticism; facing,Toblerone's decision to reduce the weight of t...,UK,2016-11 (exact date not specified),"Toblerone, Mondelez International, British con...",To reduce costs due to higher prices for ingre...,By reducing the weight of two bars from 400-gr...
6,Chicago Cubs players celebrate on the field af...,reacts to the Cubs' World Series win,Progressive Field,"November 03, 2016",Sports world; Chicago Cubs,Chicago Cubs are World Series champions; World...,innings; 8-7 in 10 innings,The Chicago Cubs won the World Series for the ...,"Progressive Field in Cleveland, Ohio","November 3, 2016","Chicago Cubs, Cleveland Indians, various sport...",Not mentioned,The Cubs edged out the Indians 8-7 in 10 innings
7,"Media caption Trump and Obama play nice, but i...",meet at White House,White House,Thursday,Donald Trump; Barack Obama,transition talks,,Transition talks between President-elect Donal...,"The White House, Washington D.C.","Thursday, after Trump's shock defeat of Hillar...",Donald Trump and Barack Obama,To facilitate a smooth transition and ensure T...,They had an 'excellent' and 'wide-ranging' con...
8,A gunman opened fire in downtown Seattle on We...,opened fire; Trump protest march in Seattle ma...,downtown Seattle,Wednesday night,A gunman,some type of personal argument,near; fire,"A shooting occurred in downtown Seattle, wound...","Downtown Seattle, near Third Avenue and Pine s...",Wednesday night (no specific date mentioned),"A gunman, Seattle police assistant chief Rober

In [30]:
# gets articles and their extracted components from the spreadsheets and compares extracted components with the true (annotated) components
df = pd.read_excel("news_5w1h_llama3.xlsx")
df = df.fillna("")
data_list2 = df.to_dict(orient="records")
e = evaluate(data_list2)
df_e = pd.DataFrame(e)
df_e['model'] = 'llama3.1'
df_e.to_pickle("news_5w1h_llama3_avaliacao.pkl")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tensor([0.8481, 0.0000, 0.8416, 0.9200, 0.8130, 0.7999])
System level F1 score: 0.704


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tensor([0.8655, 0.9060, 0.7527, 0.9257, 0.8514, 0.8681])
System level F1 score: 0.862


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

tensor([0.8989, 0.8626, 0.9134, 0.8821, 0.8555, 0.8333])
System level F1 score: 0.874


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tensor([0.9201, 0.0000, 0.8378, 0.9216, 0.0000, 0.8077])
System level F1 score: 0.581


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tensor([0.8743, 0.9569, 0.7788, 1.0000, 0.8370, 0.8676])
System level F1 score: 0.886


In [31]:
df_e

,0,1,2,3,4,5,model
0,{'text': 'FBI Director James B. Comey. (MICHAE...,[The FBI's review of new emails related to Hil...,"[Clinton's emails, , Sunday, James B. Comey; C...","[tensor(0.8191), tensor(0.), tensor(0.7678), t...","[tensor(0.8792), tensor(0.), tensor(0.9312), t...","[tensor(0.8481), tensor(0.), tensor(0.8416), t...",llama3.1
1,{'text': 'After the defeat Hillary superfan Ka...,[Celebrities stage a protest and express their...,[protest outside Trump Tower after US election...,"[tensor(0.8828), tensor(0.8831), tensor(0.7454...","[tensor(0.8488), tensor(0.9301), tensor(0.7601...","[tensor(0.8655), tensor(0.9060), tensor(0.7527...",llama3.1
2,{'text': 'Her campaign said the seemingly posi...,[Hillary Clinton blaming F.B.I. Director James...,"[Blames F.B.I. Director for Election Loss, Sat...","[tensor(0.9174), tensor(0.8038), tensor(0.), t...","[tensor(0.9408), tensor(0.8234), tensor(0.), t...","[tensor(0.9289), tensor(0.8135), tensor(0.), t...",llama3.1
3,{'text': 'The death toll from a powerful Talib...,[Truck bombing at the German consulate in Maza...,"[attacks German consulate; truck bombing, nort...","[tensor(0.8326), tensor(0.8790), tensor(0.8234...","[tensor(0.9030), tensor(0.8948), tensor(0.8771...","[tensor(0.8664), tensor(0.8868), tensor(0.8494...",llama3.1
4,{'text': 'Police investigating the Croydon tra...,"[Croydon tram crash that killed seven people, ...","[name final three victims; died, Croydon, Wedn...","[tensor(0.8111), tensor(0.8085), tensor(0.7926...","[tensor(0.8386), tensor(0.8860), tensor(0.8705...","[tensor(0.8247), tensor(0.8455), tensor(0.8298...",llama3.1
5,{'text': 'It's a triangle offense. Toblerone i...,[Toblerone's decision to reduce the weight of ...,"[are pissed, UK, , Toblerone; People, higher c...","[tensor(0.7917), tensor(1.0000), tensor(0.), t...","[tensor(0.8403), tensor(1.0000), tensor(0.), t...","[tensor(0.8153), tensor(1.0000), tensor(0.), t...",llama3.1
6,{'text': 'Chicago Cubs players celebrate on th...,[The Chicago Cubs won the World Series for the...,"[reacts to the Cubs' World Series win, Progres...","[tensor(0.8810), tensor(0.8385), tensor(0.9828...","[tensor(0.9262), tensor(0.9213), tensor(0.9828...","[tensor(0.9030), tensor(0.8780), tensor(0.9828...",llama3.1
7,{'text': 'Media caption Trump and Obama play n...,[Transition talks between President-elect Dona...,"[meet at White House, White House, Thursday, D...","[tensor(0.8576), tensor(0.8540), tensor(0.7863...","[tensor(0.9309), tensor(0.9119), tensor(0.8506...","[tensor(0.8927), tensor(0.8820), tensor(0.8172...",llama3.1
8,{'text': 'A gunman opened fire in downtown Sea...,"[A shooting occurred in downtown Seattle, woun...",[opened fire; Trump protest march in Seattle m...,"[tensor(0.8986), tensor(0.7922), tensor(0.8626...","[tensor(0.8992), tensor(0.9467), tensor(0.9705...","[tensor(0.8989), tensor(0.8626), tensor(0.9134...",llama3.1
9,{'text': 'Hillary Clinton received an unexpect...,[FBI announcement on Hillary Clinton's use of ...,[no evidence of criminal wrongdoing in her use...,"[tensor(0.9272), tensor(0.), tensor(0.8269), t...","[tensor(0.9131), tensor(0.), tensor(0.8490), t...","[tensor(0.9201), tensor(0.), tensor(0.8378), t...",llama3.1
